# Notebook 1 — Data Exploration
**Paper:** Workforce Scaling with AUM in Hedge Funds | **Author:** Rudra Jena, CFM LLP

We model headcount as $N_s = C \cdot \mathcal{A}^{\alpha}$.  
In log-log space the slope $\alpha$ is the labour-AUM elasticity:
- $\alpha < 1$: scale economies (quant funds)
- $\alpha \approx 1$: proportional deployment (pod-shop funds)


In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os, warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Path resolution (works from powerlawhf/ or notebooks/) ──────────────
_here = os.path.abspath('.')
DATDIR = (_here + '/data') if os.path.isdir(_here + '/data') else (_here + '/../data')
FIGDIR = (_here + '/figures') if os.path.isdir(_here + '/figures') else (_here + '/../figures')
os.makedirs(FIGDIR, exist_ok=True)
print(f'Data dir : {os.path.abspath(DATDIR)}')
print(f'Figure dir: {os.path.abspath(FIGDIR)}')


Data dir : /home/user/powerlawv2/data
Figure dir: /home/user/powerlawv2/figures


In [2]:
df_in  = pd.read_csv(os.path.join(DATDIR, 'insample_funds.csv'))
df_oos = pd.read_csv(os.path.join(DATDIR, 'oos_funds.csv'))
for df in [df_in, df_oos]:
    df['audited'] = df['audited'].astype(str).str.lower().isin(['true','1','yes'])
print(f'In-sample:  {len(df_in)} obs, {df_in["fund"].nunique()} funds')
print(f'OOS:        {len(df_oos)} obs, {df_oos["fund"].nunique()} funds')
CMAP  = {'Q':'#2166ac','P':'#d6604d','H':'#4dac26','M':'#7b3294','F':'#b35806','A':'#542788'}
MMAP  = {'Q':'^','P':'o','H':'s','M':'D','F':'v','A':'p'}
LABEL = {'Q':'Quant','P':'Pod-shop','H':'Hybrid','M':'Macro','F':'Fundamental L/S','A':'Activist'}


In-sample:  92 obs, 15 funds
OOS:        121 obs, 28 funds


## 1.1 Log-log scatter (in-sample, 15 funds)

In [3]:
from matplotlib.lines import Line2D
fig, ax = plt.subplots(figsize=(8,5.5))
for fund, g in df_in.groupby('fund'):
    s   = g['strategy'].iloc[0]
    aud = g['audited'].any()
    ax.scatter(g['aum_bn'], g['headcount'],
               color=CMAP.get(s,'grey'),
               marker='*' if aud else MMAP.get(s,'o'),
               s=80 if aud else 30, zorder=3,
               edgecolors='k' if aud else 'none', linewidths=0.5)
    x = np.log(g['aum_bn'].values); y = np.log(g['headcount'].values)
    a  = np.cov(x,y)[0,1]/np.var(x)
    C  = np.exp(y.mean()-a*x.mean())
    xr = np.logspace(np.log10(g['aum_bn'].min()*0.8), np.log10(g['aum_bn'].max()*1.3),80)
    ax.plot(xr, C*xr**a, color=CMAP.get(s,'grey'), lw=0.9, ls='--', alpha=0.55)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('AUM (USD billion)'); ax.set_ylabel('Headcount')
ax.set_title('Log-log scatter: headcount vs. AUM (in-sample)')
ax.legend(handles=[Line2D([0],[0],marker=MMAP[s],color='w',markerfacecolor=CMAP[s],
          markersize=8,label=LABEL[s]) for s in ['Q','P','H']], loc='upper left')
ax.grid(True,which='both',alpha=0.2)
plt.tight_layout()
plt.savefig(os.path.join(FIGDIR,'nb1_loglog.pdf'),bbox_inches='tight')
plt.show(); print('Saved.')


Saved.


## 1.2 AUM-year coverage heatmap (sorted by α̂)

In [4]:
import matplotlib.colors
fund_order = (df_in.groupby('fund')
    .apply(lambda g: np.cov(np.log(g['aum_bn']),np.log(g['headcount']))[0,1]/np.var(np.log(g['aum_bn'])))
    .sort_values().index.tolist())
years = sorted(df_in['year'].unique())
heat = pd.DataFrame(index=fund_order, columns=years, dtype=float)
for fund, g in df_in.groupby('fund'):
    for _, row in g.iterrows():
        heat.loc[fund, row['year']] = row['aum_bn']
fig, ax = plt.subplots(figsize=(11,5))
im = ax.imshow(heat.values.astype(float), aspect='auto', cmap='YlOrRd',
               norm=matplotlib.colors.LogNorm(vmin=1,vmax=300))
ax.set_xticks(range(len(years))); ax.set_xticklabels(years,rotation=45,fontsize=8)
ax.set_yticks(range(len(fund_order)))
ax.set_yticklabels([f.replace('Management','Mgmt').replace('Technologies','Tech') for f in fund_order],fontsize=8)
plt.colorbar(im,ax=ax,label='AUM (USD B)')
ax.set_title('Coverage heatmap — AUM by fund-year (sorted by α̂)')
plt.tight_layout()
plt.savefig(os.path.join(FIGDIR,'nb1_heatmap.pdf'),bbox_inches='tight')
plt.show()


## 1.3 Summary statistics

In [5]:
print('=== In-sample ===')
print(df_in[['aum_bn','headcount']].describe().round(1))
print('\n=== By strategy ===')
print(df_in.groupby('strategy')[['aum_bn','headcount']].mean().round(1))
print('\n=== OOS ===')
print(df_oos[['aum_bn','headcount']].describe().round(1))


=== In-sample ===
       aum_bn  headcount
count    92.0       92.0
mean     43.2     1375.6
std      43.2     1022.4
min       2.0      120.0
25%      13.9      687.5
50%      27.0     1254.5
75%      58.0     1710.2
max     226.0     6000.0

=== By strategy ===
          aum_bn  headcount
strategy                   
H           38.3     1261.1
P           22.4     1677.5
Q           67.5     1237.3

=== OOS ===
       aum_bn  headcount
count   121.0      121.0
mean     22.8      266.1
std      22.0      188.6
min       2.0       30.0
25%      10.0      130.0
50%      18.0      210.0
75%      29.0      350.0
max     155.0      950.0


## 1.4 Strategy distribution

In [6]:
fig, axes = plt.subplots(1,2,figsize=(11,4))
for ax, df, title in [(axes[0],df_in,'In-sample (15 funds)'),
                       (axes[1],df_oos,'OOS (26 funds)')]:
    counts = df.groupby('fund')['strategy'].first().value_counts()
    colors = [CMAP.get(s,'grey') for s in counts.index]
    bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='k',lw=0.5)
    ax.bar_label(bars)
    ax.set_xlabel('Strategy'); ax.set_ylabel('Number of funds')
    ax.set_title(title)
    ax.set_xticklabels([LABEL.get(s,s) for s in counts.index],rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(FIGDIR,'nb1_strategy_dist.pdf'),bbox_inches='tight')
plt.show()
